In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# =========================
# CUSTOM PREPROCESSING LAYER (fixes Lambda serialization issue)
# =========================
@tf.keras.utils.register_keras_serializable()  # ← This makes it saveable/loadable
class ResNet50Preprocessor(layers.Layer):
    def call(self, x):
        return tf.keras.applications.resnet50.preprocess_input(x)

# =========================
# LOAD DATA
# =========================
train_ds = keras.utils.image_dataset_from_directory(
    r"C:\Users\paula\OneDrive\Documentos\GitHub\Smart-Waste-Classification-NN-PROJECT-\data_processed\train",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    r"C:\Users\paula\OneDrive\Documentos\GitHub\Smart-Waste-Classification-NN-PROJECT-\data_processed\val",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
np.save("classes2.npy", class_names)
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# =========================
# MODEL (ResNet50)
# =========================
base_model = keras.applications.ResNet50(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

model = keras.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),

    ResNet50Preprocessor(),  # ✅ Replaces layers.Lambda(...)

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, verbose=1)
    ]
)

model.save("waste_classifier_resnet50.keras")
print("✅ ResNet50 Model saved")


# paula ---> we need to rise the epochs to 15 and make early stop

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 300 files belonging to 4 classes.
Using 60 files for validation.
Epoch 1/2
140/140 ━━━━━━━━━━━━━━━━━━━━ 99s 669ms/step - accuracy: 0.8179 - loss: 0.4955 - val_accuracy: 1.0000 - val_loss: 0.1099
Epoch 2/2
140/140 ━━━━━━━━━━━━━━━━━━━━ 97s 694ms/step - accuracy: 0.9451 - loss: 0.1690 - val_accuracy: 1.0000 - val_loss: 0.0478
Restoring model weights from the end of the best epoch: 2.
✅ ResNet50 Model saved


In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

IMG_SIZE = (224, 224)

# =========================
# MUST redefine the custom layer before loading
# =========================
@tf.keras.utils.register_keras_serializable()  # ✅ Fixed for older TF versions
class ResNet50Preprocessor(layers.Layer):
    def call(self, x):
        return tf.keras.applications.resnet50.preprocess_input(x)

# =========================
# LOAD MODEL + CLASSES
# =========================
model = tf.keras.models.load_model("waste_classifier_resnet50.keras")
class_names = np.load("classes2.npy", allow_pickle=True)

# =========================
# PREDICT FUNCTION
# =========================
def predict_image(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)

    class_index = np.argmax(preds)
    confidence = np.max(preds)

    print("-" * 30)
    print(f"Predicted Class: {class_names[class_index]}")
    print(f"Confidence:      {confidence:.2%}")
    print("-" * 30)

# =========================
# TEST
# =========================
predict_image(r"C:\Users\paula\Downloads\plastic==.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
------------------------------
Predicted Class: plastic
Confidence:      97.20%
------------------------------


In [ ]:
nn